# Scaled Dot-Product Attention

## 学习目标

能够从 Q、K、V 推导注意力分数，应用 padding/causal mask，并验证权重归一化。


## 概念模型与执行路径

Attention 用 query 与 key 的相似度决定如何汇总 value。除以 sqrt(d_k) 可控制高维点积方差；mask 在 softmax 前把不可见位置设为负无穷。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import torch
from common.models import ScaledDotProductAttention
torch.manual_seed(42)
query = torch.randn(2, 4, 8)
key = torch.randn(2, 4, 8)
value = torch.randn(2, 4, 6)
output, weights = ScaledDotProductAttention()(query, key, value)
print("scores -> weights:", weights.shape, "output:", output.shape)
print("row sums:", weights.sum(dim=-1))


### 实验 3


In [ ]:
causal_mask = torch.ones(4, 4, dtype=torch.bool).tril().unsqueeze(0)
masked_output, masked_weights = ScaledDotProductAttention()(query, key, value, causal_mask)
print(masked_weights[0])
print("future attention mass:", masked_weights.triu(diagonal=1).sum().item())


### 实验 4


In [ ]:
from torch import nn
multihead = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
tokens = torch.randn(2, 4, 8)
multi_output, multi_weights = multihead(tokens, tokens, tokens, need_weights=True)
print(multi_output.shape, multi_weights.shape)


### 实验 5


In [ ]:
# python 07-deep-learning/pytorch/examples/attention_demo.py --quick


## 底层机制

多头注意力先把 embedding 投影到多个子空间，各头独立计算注意力后拼接。padding mask 和 causal mask 解决不同问题：前者忽略补齐 token，后者阻止看到未来 token。


## 检查点

若 value 维度为 6，为什么输出最后一维是 6，而注意力权重最后两维由 query/key token 数决定？


## 试一试

构造一个 padding mask，使最后两个 key 不可见；验证这些列的权重严格为 0。


## 常见错误与调试

mask 方向取反、softmax 维度错误、忘记缩放、把 attention 权重误当成可靠解释。
